# Lab Assignment 4

MST Dependency Parsing and Training

**Total: 20 marks**

## Objective
Implement core components of an MST dependency parser, analyse a Chu-Liu/Edmonds cycle, and perform one structured-perceptron update using gold and predicted dependency trees.

## Submission
- Complete all `TODO` sections and all markdown answers.
- Do not modify the supplied data or expected-test cells.
- Run **Kernel > Restart & Run All** before submitting the completed `.ipynb` file.

## Valid dependency tree conditions
Each non-root word has exactly one incoming head; `ROOT` has none; all words are reachable from `ROOT`; and no directed cycle occurs.

In [1]:
from itertools import product
from collections import defaultdict, Counter
import pandas as pd
import matplotlib.pyplot as plt

## Task 1: Validate Dependency Trees (2 marks)

Implement `is_valid_tree()` to verify whether a set of dependency arcs forms a valid rooted dependency tree.

A valid dependency tree must satisfy all of the following conditions:

1. `ROOT` has no incoming arc.
2. Every non-root word has exactly one incoming head.
3. No word can be its own head (no self-loop).
4. The graph must not contain a directed cycle.
5. Every word must be reachable from `ROOT`.
6. The number of arcs must be exactly `number_of_tokens - 1`.

### Your task

Complete the `is_valid_tree()` function below.

Do not change the test cases.

Your function should return:

- `True` for a valid dependency tree.
- `False` for an invalid dependency structure.

You should ensure that the function correctly detects duplicate heads, missing heads, cycles, self-loops, disconnected components, and arcs entering `ROOT`.

In [2]:
def is_valid_tree(tokens, arcs, root="ROOT"):
    # TODO: Implement all rooted dependency-tree checks.
    non_root = [w for w in tokens if w != root]

    # The tree must contain exactly one incoming arc for each non-root node.
    if len(arcs) != len(non_root):
        return False

    # Reject duplicate arcs.
    if len(set(arcs)) != len(arcs):
        return False

    heads = {}
    token_set = set(tokens)

    for head, dep in arcs:
        # Every endpoint must be a known token.
        if head not in token_set or dep not in token_set:
            return False

        # No self-loop and no edge may enter ROOT.
        if head == dep or dep == root:
            return False

        # Every non-root node must have exactly one incoming head.
        if dep in heads:
            return False

        heads[dep] = head

    # Every non-root token must appear exactly once as a dependent.
    if set(heads) != set(non_root):
        return False

    # Starting from every non-root node, follow parent pointers.
    # A missing parent means the node is disconnected; a repeated
    # node means there is a directed cycle.
    for start in non_root:
        current = start
        seen = set()

        while current != root:
            if current in seen:
                return False
            if current not in heads:
                return False

            seen.add(current)
            current = heads[current]

    return True

test_tokens = ["ROOT", "A", "B", "C"]
valid = [("ROOT","A"), ("A","B"), ("A","C")]
cycle = [("ROOT","A"), ("B","C"), ("C","B")]
duplicate = [("ROOT","A"), ("A","B"), ("C","B")]
print(is_valid_tree(test_tokens, valid))      # Expected: True
print(is_valid_tree(test_tokens, cycle))      # Expected: False
print(is_valid_tree(test_tokens, duplicate))  # Expected: False

True
False
False


## Task 2: Exhaustive MST Reference Parser (3 marks)

Implement an exhaustive reference parser for a small dependency graph.

For every non-root word:

1. Enumerate all possible incoming heads.
2. Construct candidate dependency trees.
3. Use `is_valid_tree()` from Task 1 to reject invalid structures.
4. Compute the total score of every valid tree.
5. Return the highest-scoring tree and its score.

The score of a tree is:

$$
Score(G)=\sum_{(h,d)\in G} Score(h\rightarrow d)
$$

where:

- $h$ = head
- $d$ = dependent
- $G$ = dependency tree

### Your task

Complete:

```python
exhaustive_mst(tokens, score)

In [3]:

tokens = ["ROOT", "Researchers", "analyze", "data", "carefully"]

score = {
    ("ROOT","Researchers"):1,
    ("ROOT","analyze"):7,
    ("ROOT","data"):1,
    ("ROOT","carefully"):1,

    ("Researchers","analyze"):2,
    ("Researchers","data"):2,
    ("Researchers","carefully"):2,

    ("analyze","Researchers"):6,
    ("analyze","data"):7,
    ("analyze","carefully"):6,

    ("data","Researchers"):2,
    ("data","analyze"):3,
    ("data","carefully"):3,

    ("carefully","Researchers"):2,
    ("carefully","analyze"):3,
    ("carefully","data"):4,
}

def exhaustive_mst(tokens, score, root="ROOT"):
    # TODO:
    # 1. Generate candidate heads for every non-root node.
    # 2. Construct candidate trees.
    # 3. Keep only valid trees using is_valid_tree().
    # 4. Calculate total tree score.
    # 5. Return the highest-scoring tree.

    non_root = [word for word in tokens if word != root]
    choices = [
        [head for head in tokens if head != dep]
        for dep in non_root
    ]

    best_score = float("-inf")
    best_arcs = None

    for heads in product(*choices):
        candidate_arcs = list(zip(heads, non_root))

        if is_valid_tree(tokens, candidate_arcs, root):
            total_score = sum(score[arc] for arc in candidate_arcs)

            if total_score > best_score:
                best_score = total_score
                best_arcs = candidate_arcs

    return best_score, best_arcs

best_score, best_arcs = exhaustive_mst(tokens, score)

print("Best score:", best_score)
print("Best arcs:", best_arcs)

Best score: 26
Best arcs: [('analyze', 'Researchers'), ('ROOT', 'analyze'), ('analyze', 'data'), ('analyze', 'carefully')]


## Task 3: Chu-Liu/Edmonds — Best Incoming Edges and Cycle Detection (3 marks)

The Chu-Liu/Edmonds algorithm starts by selecting the highest-scoring incoming edge for every non-root node.

For each node $v$:

$$
parent(v)
=
\arg\max_{u\neq v} Score(u\rightarrow v)
$$

If the selected edges form a valid tree, the algorithm is finished.

However, if the selected edges contain a directed cycle, the cycle must be detected and contracted before continuing.

### Your task

Complete the following two functions:

1. `best_incoming()`
2. `find_cycle()`

`best_incoming()` should select the highest-scoring incoming edge for every non-root node.

`find_cycle()` should:

- return one directed cycle as a list of nodes if a cycle exists;
- return `None` if no cycle exists.

### Questions

After running the code:

1. What edges were selected by the greedy incoming-edge step?
2. Which nodes form the cycle?
3. Why is the greedy result not a valid dependency tree?
4. Why can we not simply accept the greedy result as the MST?

In [4]:
cycle_tokens = ["ROOT", "A", "B", "C"]

cycle_score = {
    ("ROOT","A"):5,
    ("ROOT","B"):4,
    ("ROOT","C"):3,

    ("A","B"):10,
    ("B","A"):10,

    ("A","C"):7,
    ("B","C"):8,

    ("C","A"):2,
    ("C","B"):2,
}


def best_incoming(tokens, score, root="ROOT"):
    # TODO:
    # For every non-root node:
    #   find the incoming edge with maximum score
    # Do not allow edges entering ROOT.

    selected_arcs = []

    for dep in tokens:
        if dep == root:
            continue

        candidates = [
            (head, dep, score[(head, dep)])
            for head in tokens
            if head != dep
        ]

        if not candidates:
            raise ValueError(f"No incoming edge found for {dep}")

        best_head, _, _ = max(candidates, key=lambda item: item[2])
        selected_arcs.append((best_head, dep))

    return selected_arcs


def find_cycle(arcs):
    # TODO:
    # Detect a directed cycle in the selected arcs.
    # Return the cycle as a list of nodes.
    # Return None if no cycle exists.

    parent = {dependent: head for head, dependent in arcs}
    visited_global = set()

    for start in parent:
        if start in visited_global:
            continue

        path = []
        position = {}
        current = start

        while (
            current != "ROOT"
            and current in parent
            and current not in position
            and current not in visited_global
        ):
            position[current] = len(path)
            path.append(current)
            current = parent[current]

        visited_global.update(path)

        if current in position:
            return path[position[current]:]

    return None

greedy_arcs = best_incoming(cycle_tokens, cycle_score)

print("Greedy arcs:", greedy_arcs)
print("Cycle:", find_cycle(greedy_arcs))

Greedy arcs: [('B', 'A'), ('A', 'B'), ('B', 'C')]
Cycle: ['A', 'B']


**Task 3 explanation:**

1. The greedy incoming-edge step selects `B -> A`, `A -> B`, and `B -> C`.
2. The directed cycle is `A -> B -> A`, so the cycle nodes are `{A, B}`.
3. The greedy result is not a valid dependency tree because `A` and `B` form a directed cycle and therefore do not have a valid rooted path from `ROOT`.
4. We cannot accept the greedy result as the MST because independently maximizing each node's incoming edge does not enforce the global tree constraint. Chu-Liu/Edmonds resolves such cycles by contracting them and comparing the appropriate adjusted alternatives.


## Task 4: Chu-Liu/Edmonds Cycle Contraction (3 marks)

Suppose the greedy step produces the cycle:

$$
A\rightarrow B
$$

and

$$
B\rightarrow A
$$

with:

$$
Score(A\rightarrow B)=10
$$

$$
Score(B\rightarrow A)=10
$$

The total score of the selected cycle is:

$$
CycleScore=10+10=20
$$

When an external edge enters a node inside the cycle, the cycle is contracted into a single **supernode**.

For an external edge $h\rightarrow d$ entering the cycle, calculate the adjusted score as:

$$
\boxed{
AdjustedScore(h\rightarrow d)
=
Score(h\rightarrow d)
+
CycleScore
-
Score(old\_incoming\_edge(d))
}
$$

### Example

For:

$$
ROOT\rightarrow A
$$

we have:

$$
Score(ROOT\rightarrow A)=5
$$

The selected edge entering `A` inside the cycle is:

$$
B\rightarrow A
$$

with score:

$$
Score(B\rightarrow A)=10
$$

Therefore:

$$
AdjustedScore(ROOT\rightarrow A)
=
5+20-10
=
15
$$

### Your task

Complete `entering_cycle_table()`.

Create a table for:

- `ROOT -> A`
- `ROOT -> B`

The table must contain:

- candidate edge
- original score
- selected cycle edge being removed
- removed edge score
- total cycle score
- adjusted score

### Interpretation question

Which candidate edge should be selected to enter the contracted cycle?

Explain which original cycle edge will be removed when the contracted tree is expanded back to the original graph.

In [5]:
def entering_cycle_table(score, cycle, selected_cycle_arcs):
    # TODO:
    # Create a DataFrame containing:
    #   outside_head
    #   cycle_dependent
    #   original_score
    #   removed_cycle_arc
    #   removed_score
    #   cycle_total
    #   adjusted_score

    selected_parent = {
        dependent: head
        for head, dependent in selected_cycle_arcs
    }

    cycle_total = sum(score[arc] for arc in selected_cycle_arcs)
    cycle_set = set(cycle)
    rows = []

    # The supplied task specifically asks for the ROOT -> A and
    # ROOT -> B candidates that enter the contracted cycle.
    for (head, dependent), original_score in score.items():
        if head == "ROOT" and dependent in cycle_set:
            removed_arc = (selected_parent[dependent], dependent)
            removed_score = score[removed_arc]
            adjusted_score = (
                original_score
                + cycle_total
                - removed_score
            )

            rows.append({
                "outside_head": head,
                "cycle_dependent": dependent,
                "original_score": original_score,
                "removed_cycle_arc": f"{removed_arc[0]} -> {removed_arc[1]}",
                "removed_score": removed_score,
                "cycle_total": cycle_total,
                "adjusted_score": adjusted_score
            })

    return pd.DataFrame(rows)

selected_cycle_arcs = [
    ("B","A"),
    ("A","B")
]

entering_cycle_table(
    cycle_score,
    ["A", "B"],
    selected_cycle_arcs
)

,outside_head,cycle_dependent,original_score,removed_cycle_arc,removed_score,cycle_total,adjusted_score
0,ROOT,A,5,B -> A,10,20,15
1,ROOT,B,4,A -> B,10,20,14


## Task 5: Implement the Chu-Liu/Edmonds MST Parser (4 marks)

In this task, you will implement the **Chu-Liu/Edmonds algorithm from scratch** for finding the maximum-scoring rooted directed spanning tree.

Do **not** use:

- `networkx.maximum_spanning_arborescence`
- any other library implementation of Chu-Liu/Edmonds

You may use Python data structures such as dictionaries, lists, and sets.

### Algorithm

Your implementation should follow these steps:

#### Step 1: Select the best incoming edge

For every non-root node, select the incoming edge with the highest score.

#### Step 2: Check for cycles

If the selected edges form a valid rooted tree, return them.

If a directed cycle exists, continue.

#### Step 3: Identify the cycle

Find one directed cycle:

$$
C=\{v_1,v_2,\ldots,v_k\}
$$

#### Step 4: Contract the cycle

Replace all nodes in the cycle by a single supernode.

For an edge entering the cycle:

$$
u\rightarrow v,\qquad v\in C
$$

use:

$$
w'(u\rightarrow C)
=
w(u\rightarrow v)
+
W_C
-
w(parent(v)\rightarrow v)
$$

where $W_C$ is the total score of the selected cycle edges.

For an edge leaving the cycle:

$$
v\rightarrow u,\qquad v\in C
$$

keep the maximum-scoring corresponding outgoing edge.

#### Step 5: Recursively solve the contracted graph

Run the same procedure on the contracted graph.

#### Step 6: Expand the cycle

Use the selected incoming edge to determine which cycle edge must be removed.

Restore the remaining cycle edges and return the final maximum spanning arborescence.

### Required functions

Complete:

```python
chu_liu_edmonds()

In [6]:

def select_best_incoming(nodes, edges, root="ROOT"):
    """
    Select the highest-scoring incoming edge for every non-root node.

    edges:
        dictionary {(head, dependent): score}
    """
    # TODO
    selected_arcs = []

    for dependent in nodes:
        if dependent == root:
            continue

        candidates = [
            (head, dependent, weight)
            for (head, dep), weight in edges.items()
            if dep == dependent and head != dependent
        ]

        if not candidates:
            raise ValueError(
                f"No incoming edge available for node {dependent}"
            )

        best_head, _, _ = max(
            candidates,
            key=lambda item: item[2]
        )
        selected_arcs.append((best_head, dependent))

    return selected_arcs

def detect_cycle(arcs, root="ROOT"):
    """
    Return one directed cycle from arcs.
    Return None if no cycle exists.
    """
    # TODO

    parent = {
        dependent: head
        for head, dependent in arcs
    }

    visited_global = set()

    for start in parent:
        if start in visited_global:
            continue

        path = []
        position = {}
        current = start

        while (
            current != root
            and current in parent
            and current not in position
            and current not in visited_global
        ):
            position[current] = len(path)
            path.append(current)
            current = parent[current]

        visited_global.update(path)

        if current in position:
            return path[position[current]:]

    return None


def contract_cycle(nodes, edges, selected_arcs, cycle, root="ROOT"):
    """
    Contract the detected cycle into a single supernode.

    Return:
        contracted_nodes
        contracted_edges
        mapping_information

    mapping_information is needed later to expand
    the contracted solution back to the original graph.
    """
    # TODO

    cycle_set = set(cycle)

    # The supernode is only an internal representation used during
    # recursive contraction. It cannot collide with an ordinary token.
    supernode = (
        "__CYCLE__",
        tuple(sorted(repr(node) for node in cycle_set))
    )

    contracted_nodes = [
        node for node in nodes
        if node not in cycle_set
    ]
    contracted_nodes.append(supernode)

    selected_parent = {
        dependent: head
        for head, dependent in selected_arcs
    }

    cycle_arcs = [
        arc for arc in selected_arcs
        if arc[0] in cycle_set and arc[1] in cycle_set
    ]

    cycle_total = sum(edges[arc] for arc in cycle_arcs)

    contracted_edges = {}
    incoming_options = defaultdict(list)
    outgoing_options = defaultdict(list)

    for (head, dependent), weight in edges.items():

        # Internal cycle edges disappear after contraction.
        if head in cycle_set and dependent in cycle_set:
            continue

        # Edge entering the cycle:
        # add the cycle total and subtract the selected
        # incoming cycle edge of the node that it enters.
        if head not in cycle_set and dependent in cycle_set:
            removed_arc = (
                selected_parent[dependent],
                dependent
            )
            adjusted_weight = (
                weight
                + cycle_total
                - edges[removed_arc]
            )

            key = (head, supernode)

            incoming_options[key].append(
                (
                    adjusted_weight,
                    (head, dependent),
                    removed_arc,
                    weight
                )
            )

        # Edge leaving the cycle keeps its original score.
        elif head in cycle_set and dependent not in cycle_set:
            key = (supernode, dependent)

            outgoing_options[key].append(
                (
                    weight,
                    (head, dependent)
                )
            )

        # Edge outside the cycle is unchanged.
        else:
            contracted_edges[(head, dependent)] = weight

    incoming_map = {}

    for key, candidates in incoming_options.items():
        best = max(
            candidates,
            key=lambda item: item[0]
        )

        contracted_edges[key] = best[0]

        incoming_map[key] = {
            "original_arc": best[1],
            "removed_cycle_arc": best[2],
            "original_score": best[3]
        }

    outgoing_map = {}

    for key, candidates in outgoing_options.items():
        best = max(
            candidates,
            key=lambda item: item[0]
        )

        contracted_edges[key] = best[0]

        outgoing_map[key] = {
            "original_arc": best[1]
        }

    mapping_information = {
        "supernode": supernode,
        "cycle": cycle_set,
        "cycle_total": cycle_total,
        "selected_cycle_arcs": cycle_arcs,
        "incoming_map": incoming_map,
        "outgoing_map": outgoing_map
    }

    return (
        contracted_nodes,
        contracted_edges,
        mapping_information
    )

def expand_cycle(
    selected_contracted_arcs,
    cycle,
    mapping_information,
    original_selected_arcs
):
    """
    Expand the contracted solution back to the original graph.

    The incoming edge selected for the supernode determines
    which cycle edge must be removed.
    """
    # TODO

    supernode = mapping_information["supernode"]
    incoming_map = mapping_information["incoming_map"]
    outgoing_map = mapping_information["outgoing_map"]
    selected_cycle_arcs = mapping_information["selected_cycle_arcs"]

    expanded_arcs = []
    removed_cycle_arc = None

    for arc in selected_contracted_arcs:

        # An external edge entering the supernode tells us
        # which original cycle edge must be removed.
        if arc[1] == supernode:
            info = incoming_map[arc]
            expanded_arcs.append(info["original_arc"])
            removed_cycle_arc = info["removed_cycle_arc"]

        # Restore the original edge for an edge leaving the cycle.
        elif arc[0] == supernode:
            info = outgoing_map[arc]
            expanded_arcs.append(info["original_arc"])

        # Ordinary outside-to-outside edge.
        else:
            expanded_arcs.append(arc)

    # A valid contracted arborescence must enter the contracted cycle
    # exactly once. Remove that cycle edge and keep the rest.
    if removed_cycle_arc is None:
        raise ValueError(
            "The contracted cycle was not entered during expansion."
        )

    for arc in selected_cycle_arcs:
        if arc != removed_cycle_arc:
            expanded_arcs.append(arc)

    return expanded_arcs

def chu_liu_edmonds(tokens, score, root="ROOT"):
    """
    Implement the Chu-Liu/Edmonds maximum spanning
    arborescence algorithm from scratch.

    Return:
        total_score, final_arcs
    """

    # TODO 1:
    # Select the highest-scoring incoming edge
    # for every non-root node.

    # TODO 2:
    # Detect whether the selected edges contain a cycle.

    # TODO 3:
    # If there is no cycle, return the selected tree.

    # TODO 4:
    # If there is a cycle, contract it.

    # TODO 5:
    # Recursively solve the contracted graph.

    # TODO 6:
    # Expand the cycle.

    def solve(nodes, edges):
        selected_arcs = select_best_incoming(
            nodes,
            edges,
            root
        )

        cycle = detect_cycle(
            selected_arcs,
            root
        )

        if cycle is None:
            return selected_arcs

        (
            contracted_nodes,
            contracted_edges,
            mapping_information
        ) = contract_cycle(
            nodes,
            edges,
            selected_arcs,
            cycle,
            root
        )

        selected_contracted_arcs = solve(
            contracted_nodes,
            contracted_edges
        )

        return expand_cycle(
            selected_contracted_arcs,
            cycle,
            mapping_information,
            selected_arcs
        )

    final_arcs = solve(
        list(tokens),
        dict(score)
    )

    total_score = sum(
        score[arc]
        for arc in final_arcs
    )

    return total_score, final_arcs

# Test on the supplied graph
cle_score, cle_arcs = chu_liu_edmonds(tokens, score)

print("Chu-Liu/Edmonds score:", cle_score)
print("Chu-Liu/Edmonds arcs:", cle_arcs)

print(
    "Matches exhaustive MST score?",
    cle_score == best_score
)

Chu-Liu/Edmonds score: 26
Chu-Liu/Edmonds arcs: [('analyze', 'Researchers'), ('ROOT', 'analyze'), ('analyze', 'data'), ('analyze', 'carefully')]
Matches exhaustive MST score? True


## Task 6: Inference-Based Learning with Structured Perceptron (5 marks)

In this task, you will train the MST dependency parser using **inference-based structured perceptron learning**.

The parser uses the current weight vector to score every possible dependency tree and selects the highest-scoring tree:

$$
\boxed{
G_{pred}
=
\arg\max_{G\in T(x)}
\mathbf{w}\cdot\Phi(x,G)
}
$$

where:

- $x$ = input sentence
- $T(x)$ = set of valid dependency trees for the sentence
- $\mathbf{w}$ = current feature-weight vector
- $\Phi(x,G)$ = feature vector of dependency tree $G$

The predicted tree is then compared with the gold tree.

If:

$$
G_{pred}\neq G_{gold}
$$

update the weights using:

$$
\boxed{
\mathbf{w}_{new}
=
\mathbf{w}_{old}
+
\Phi(x,G_{gold})
-
\Phi(x,G_{pred})
}
$$

Equivalently, for each feature:

$$
\boxed{
w_{new}[f]
=
w_{old}[f]
+
\Phi(x,G_{gold})[f]
-
\Phi(x,G_{pred})[f]
}
$$

### Training procedure

Your training loop must perform the following steps:

1. Initialize the feature weights.
2. Take one training sentence and its gold dependency tree.
3. Compute the score of candidate dependency arcs using the current weights.
4. Run your **own Chu-Liu/Edmonds implementation from Task 5** to obtain the predicted tree.
5. Compute the feature vector of the gold tree.
6. Compute the feature vector of the predicted tree.
7. Update the weights using the structured perceptron rule.
8. Repeat for all training examples.
9. Repeat the complete process for multiple epochs.

### Important

The training procedure must call your implementation from **Task 5**.

Do not use:

```python
networkx.maximum_spanning_arborescence()

In [7]:

from collections import Counter

# ---------------------------------------------------------
# Training data
# ---------------------------------------------------------

training_data = [

    {
        "tokens": ["ROOT", "John", "saw", "Mary"],
        "pos": {
            "ROOT": "ROOT",
            "John": "NOUN",
            "saw": "VERB",
            "Mary": "NOUN"
        },
        "gold": [
            ("ROOT", "saw"),
            ("saw", "John"),
            ("saw", "Mary")
        ]
    },

    {
        "tokens": ["ROOT", "Dogs", "chase", "cats"],
        "pos": {
            "ROOT": "ROOT",
            "Dogs": "NOUN",
            "chase": "VERB",
            "cats": "NOUN"
        },
        "gold": [
            ("ROOT", "chase"),
            ("chase", "Dogs"),
            ("chase", "cats")
        ]
    },

    {
        "tokens": ["ROOT", "Alice", "likes", "Bob"],
        "pos": {
            "ROOT": "ROOT",
            "Alice": "NOUN",
            "likes": "VERB",
            "Bob": "NOUN"
        },
        "gold": [
            ("ROOT", "likes"),
            ("likes", "Alice"),
            ("likes", "Bob")
        ]
    }
]


# ---------------------------------------------------------
# Feature extraction
# ---------------------------------------------------------

def arc_features(tokens, pos, head, dep):
    """
    Return a list of feature names for one dependency arc.

    Required feature templates:
      1. head word
      2. dependent word
      3. head POS
      4. dependent POS
      5. direction
      6. distance
      7. head-dependent word pair
    """

    # TODO
    # Do not simply return the score.
    # Return feature names/counts for this arc.

    head_index = tokens.index(head)
    dep_index = tokens.index(dep)

    direction = (
        "RIGHT"
        if head_index < dep_index
        else "LEFT"
    )

    distance = abs(head_index - dep_index)

    return [
        f"HEAD_WORD={head}",
        f"DEP_WORD={dep}",
        f"HEAD_POS={pos[head]}",
        f"DEP_POS={pos[dep]}",
        f"DIRECTION={direction}",
        f"DISTANCE={distance}",
        f"PAIR={head}->{dep}",
    ]

def tree_feature_vector(tokens, pos, tree):
    """
    Sum the feature vectors of all arcs in a dependency tree.

    Return a Counter or dictionary:
        feature -> count
    """

    # TODO

    feature_counts = Counter()

    for head, dep in tree:
        feature_counts.update(
            arc_features(
                tokens,
                pos,
                head,
                dep
            )
        )

    return feature_counts

# ---------------------------------------------------------
# Arc and tree scoring
# ---------------------------------------------------------

def score_arc(tokens, pos, head, dep, weights):
    """
    Score one dependency arc:

        score(h -> d) = w . f(h -> d)

    """

    # TODO

    features = arc_features(
        tokens,
        pos,
        head,
        dep
    )

    return sum(
        weights[feature]
        for feature in features
    )


def build_score_graph(tokens, pos, weights):
    """
    Construct the complete directed graph of candidate
    dependency arcs.

    Return:
        {(head, dependent): score}
    """

    # TODO

    score_graph = {}

    for dependent in tokens:
        if dependent == "ROOT":
            continue

        for head in tokens:
            if head == dependent:
                continue

            score_graph[(head, dependent)] = score_arc(
                tokens,
                pos,
                head,
                dependent,
                weights
            )

    return score_graph


def score_tree(tokens, pos, tree, weights):
    """
    Return the total score of a dependency tree.
    """

    # TODO

    return sum(
        score_arc(
            tokens,
            pos,
            head,
            dependent,
            weights
        )
        for head, dependent in tree
    )

# ---------------------------------------------------------
# Structured perceptron update
# ---------------------------------------------------------

def perceptron_update(
    weights,
    gold_features,
    predicted_features,
    learning_rate=1.0
):
    """
    Structured perceptron update:

        w_new =
        w_old
        + eta * Phi(gold)
        - eta * Phi(predicted)
    """

    # TODO

    updated_weights = Counter(weights)

    for feature, count in gold_features.items():
        updated_weights[feature] += (
            learning_rate * count
        )

    for feature, count in predicted_features.items():
        updated_weights[feature] -= (
            learning_rate * count
        )

    return updated_weights


# ---------------------------------------------------------
# Inference-based training
# ---------------------------------------------------------

def train_perceptron(
    training_data,
    num_epochs=5,
    learning_rate=1.0
):
    """
    Train the MST parser using inference-based
    structured perceptron learning.

    For every epoch:

        1. Infer predicted tree using Task 5.
        2. Compare with gold tree.
        3. Update weights if prediction is incorrect.
        4. Record number of mistakes.
    """

    # Initialize all weights to zero.
    weights = Counter()

    history = []

    for epoch in range(num_epochs):

        mistakes = 0

        for example in training_data:

            tokens = example["tokens"]
            pos = example["pos"]
            gold_tree = example["gold"]

            # ---------------------------------------------
            # TODO 1:
            # Build arc scores using current weights.
            # ---------------------------------------------

            score_graph = build_score_graph(
                tokens,
                pos,
                weights
            )

            # ---------------------------------------------
            # TODO 2:
            # Run YOUR Task 5 implementation.
            # ---------------------------------------------

            predicted_score, predicted_tree = chu_liu_edmonds(
                tokens,
                score_graph
            )

            # ---------------------------------------------
            # TODO 3:
            # Compare predicted and gold trees.
            # ---------------------------------------------

            if set(predicted_tree) != set(gold_tree):

                mistakes += 1

                # -----------------------------------------
                # TODO 4:
                # Compute gold feature vector.
                # -----------------------------------------

                gold_features = tree_feature_vector(
                    tokens,
                    pos,
                    gold_tree
                )

                # -----------------------------------------
                # TODO 5:
                # Compute predicted feature vector.
                # -----------------------------------------

                predicted_features = tree_feature_vector(
                    tokens,
                    pos,
                    predicted_tree
                )

                # -----------------------------------------
                # TODO 6:
                # Update weights.
                # -----------------------------------------

                weights = perceptron_update(
                    weights,
                    gold_features,
                    predicted_features,
                    learning_rate
                )

        history.append(mistakes)

        print(
            f"Epoch {epoch + 1}: "
            f"{mistakes} mistake(s)"
        )

    return weights, history

In [8]:
# Run the complete inference-based training

final_weights, training_history = train_perceptron(
    training_data,
    num_epochs=5,
    learning_rate=1.0
)

print("\nTraining history:")
for epoch, mistakes in enumerate(training_history, start=1):
    print(
        f"Epoch {epoch}: "
        f"{mistakes} mistake(s)"
    )

print("\nLearned weights:")
for feature, weight in sorted(final_weights.items()):
    print(f"{feature}: {weight}")

Epoch 1: 1 mistake(s)
Epoch 2: 0 mistake(s)
Epoch 3: 0 mistake(s)
Epoch 4: 0 mistake(s)
Epoch 5: 0 mistake(s)

Training history:
Epoch 1: 1 mistake(s)
Epoch 2: 0 mistake(s)
Epoch 3: 0 mistake(s)
Epoch 4: 0 mistake(s)
Epoch 5: 0 mistake(s)

Learned weights:
DEP_POS=NOUN: 0.0
DEP_POS=VERB: 0.0
DEP_WORD=John: 0.0
DEP_WORD=Mary: 0.0
DEP_WORD=saw: 0.0
DIRECTION=LEFT: 1.0
DIRECTION=RIGHT: -1.0
DISTANCE=1: 1.0
DISTANCE=2: 0.0
DISTANCE=3: -1.0
HEAD_POS=ROOT: -2.0
HEAD_POS=VERB: 2.0
HEAD_WORD=ROOT: -2.0
HEAD_WORD=saw: 2.0
PAIR=ROOT->John: -1.0
PAIR=ROOT->Mary: -1.0
PAIR=ROOT->saw: 0.0
PAIR=saw->John: 1.0
PAIR=saw->Mary: 1.0


## Task 6: Training Analysis

### Q1. Prediction Errors

For each epoch, report the number of incorrectly predicted trees.

| Epoch | Number of Mistakes |
|------:|-------------------:|
| 1 | 1 |
| 2 | 0 |
| 3 | 0 |
| 4 | 0 |
| 5 | 0 |

### Q2. Weight Updates

Two features whose weights increased are `HEAD_POS=VERB` and `HEAD_WORD=saw`. Their weights increased because they occur in gold dependency arcs and, during the first incorrect prediction, their gold-tree feature counts were greater than their predicted-tree counts. The structured perceptron therefore added positive values to these features. For this run, both increased to `+2`.

Other positive examples include `DIRECTION=LEFT` and `DISTANCE=1`, which increased to `+1`.

### Q3. Negative Updates

Two features whose weights decreased are `HEAD_POS=ROOT` and `HEAD_WORD=ROOT`. They received negative updates because the incorrect predicted tree contained more occurrences of these features than the gold tree in the update. The perceptron subtracts the predicted feature vector, so these features became less preferred. In this run they reached `-2`.

Other negative examples include `DIRECTION=RIGHT` (`-1`) and `DISTANCE=3` (`-1`).

### Q4. Effect of the Perceptron Update

The update

$$
\mathbf{w}_{new}
=
\mathbf{w}_{old}
+
\Phi(x,G_{gold})
-
\Phi(x,G_{pred})
$$

increases the weights of features that support the gold tree and decreases the weights of features that support the predicted tree. Therefore, when the parser performs inference again, the gold dependency arcs receive higher relative scores while the incorrect predicted arcs become less attractive. This makes the gold structure more likely to be selected.

### Q5. Training Behaviour

Yes. The number of mistakes decreased from `1` in epoch 1 to `0` from epoch 2 onward. After the first update, the learned feature weights were sufficient for the parser to select the gold tree for all three training examples, so no further updates were required.